# Statistical Analysis - ShipmentSure

This notebook performs statistical tests to understand which features significantly affect on-time delivery.

**Tests performed:**
1. Descriptive Statistics
2. Skewness & Kurtosis Analysis
3. Chi-Square Test (categorical features)
4. Mann-Whitney U Test (numeric features)
5. Summary of Findings

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
print("Libraries imported!")

In [ ]:
# Load dataset
df = pd.read_csv('Train.csv')

# Drop ID column
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

print(f"Dataset shape: {df.shape}")
df.head()

## 1. Descriptive Statistics

Basic summary of all numeric features: mean, median, standard deviation, min, max.

In [ ]:
# Detailed descriptive statistics
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

desc_stats = df[numeric_cols].describe().T
desc_stats['median'] = df[numeric_cols].median()
desc_stats['range'] = desc_stats['max'] - desc_stats['min']

print("DESCRIPTIVE STATISTICS:")
print("="*80)
desc_stats[['count', 'mean', 'median', 'std', 'min', 'max', 'range']]

## 2. Skewness & Kurtosis Analysis

- **Skewness** measures asymmetry. Values close to 0 = symmetric/normal.
- **Kurtosis** measures tail heaviness. Values close to 0 = normal tails.

In [ ]:
# Calculate skewness and kurtosis for all numeric columns
skew_kurt = pd.DataFrame({
    'Skewness': df[numeric_cols].skew(),
    'Kurtosis': df[numeric_cols].kurtosis()
})

# Add interpretation
def interpret_skewness(s):
    if abs(s) < 0.5:
        return 'Approximately Symmetric'
    elif abs(s) < 1:
        return 'Moderately Skewed'
    else:
        return 'Highly Skewed'

skew_kurt['Skew_Interpretation'] = skew_kurt['Skewness'].apply(interpret_skewness)

print("SKEWNESS & KURTOSIS ANALYSIS:")
print("="*70)
print(skew_kurt.to_string())

In [ ]:
# Visualize skewness
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution & Skewness of Numeric Features', fontsize=14, fontweight='bold')

for idx, col in enumerate(numeric_cols):
    if idx >= 6:
        break
    row = idx // 3
    col_idx = idx % 3
    skew_val = df[col].skew()
    
    sns.histplot(data=df, x=col, kde=True, ax=axes[row][col_idx], 
                 color='#3498db', bins=25)
    axes[row][col_idx].set_title(f'{col}\nSkewness: {skew_val:.3f}', fontsize=10)
    
    # Add vertical line at mean
    axes[row][col_idx].axvline(df[col].mean(), color='red', linestyle='--', label='Mean')
    axes[row][col_idx].axvline(df[col].median(), color='green', linestyle='--', label='Median')
    axes[row][col_idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Chi-Square Test for Categorical Features

The chi-square test checks if there is a **statistically significant relationship** between categorical features and the target variable (on-time delivery).

- **Null Hypothesis (H0):** The feature and target are independent (no relationship)
- **Alternative Hypothesis (H1):** The feature and target are dependent (relationship exists)
- **If p-value < 0.05:** We reject H0 → significant relationship exists

In [ ]:
# Chi-Square test for categorical features vs target
cat_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
target = 'Reached.on.Time_Y.N'

print("CHI-SQUARE TEST RESULTS:")
print("="*70)
print(f"{'Feature':<25} {'Chi2':>10} {'p-value':>12} {'Significant?':>15}")
print("-"*70)

chi_results = []
for col in cat_cols:
    # Create contingency table
    contingency = pd.crosstab(df[col], df[target])
    
    # Perform chi-square test
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    
    significant = '✅ Yes' if p_value < 0.05 else '❌ No'
    chi_results.append({'Feature': col, 'Chi2': chi2, 'p-value': p_value, 'Significant': significant})
    print(f"{col:<25} {chi2:>10.4f} {p_value:>12.6f} {significant:>15}")

print("\nInterpretation: If p-value < 0.05, the feature has a significant")
print("relationship with on-time delivery.")

In [ ]:
# Visualize contingency tables
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Categorical Features vs On-Time Delivery', fontsize=14, fontweight='bold')

for idx, col in enumerate(cat_cols):
    row = idx // 2
    col_idx = idx % 2
    
    ct = pd.crosstab(df[col], df[target], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[row][col_idx], color=['#e74c3c', '#2ecc71'])
    axes[row][col_idx].set_title(f'{col} (p={chi_results[idx]["p-value"]:.4f})', fontsize=11)
    axes[row][col_idx].set_ylabel('Percentage (%)')
    axes[row][col_idx].legend(['Late', 'On-Time'])
    axes[row][col_idx].set_xticklabels(axes[row][col_idx].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Mann-Whitney U Test for Numeric Features

This test compares numeric feature distributions between the two delivery groups (late vs on-time).

- **H0:** No difference in the feature between late and on-time deliveries
- **H1:** There is a significant difference
- **If p-value < 0.05:** The feature differs significantly between groups

In [ ]:
# Mann-Whitney U test for numeric features
numeric_features = ['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product',
                     'Prior_purchases', 'Discount_offered', 'Weight_in_gms']

# Split data by target
late = df[df[target] == 0]
on_time = df[df[target] == 1]

print("MANN-WHITNEY U TEST RESULTS:")
print("="*80)
print(f"{'Feature':<25} {'Late Mean':>12} {'OnTime Mean':>12} {'U-stat':>12} {'p-value':>12} {'Sig?':>8}")
print("-"*80)

mw_results = []
for col in numeric_features:
    u_stat, p_value = stats.mannwhitneyu(late[col], on_time[col], alternative='two-sided')
    significant = '✅ Yes' if p_value < 0.05 else '❌ No'
    
    mw_results.append({
        'Feature': col, 'U_stat': u_stat, 'p_value': p_value,
        'Late_mean': late[col].mean(), 'OnTime_mean': on_time[col].mean()
    })
    
    print(f"{col:<25} {late[col].mean():>12.2f} {on_time[col].mean():>12.2f} "
          f"{u_stat:>12.0f} {p_value:>12.6f} {significant:>8}")

print("\nInterpretation: If p-value < 0.05, the feature significantly differs")
print("between late and on-time deliveries.")

In [ ]:
# Visualize group differences
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: Late vs On-Time Delivery', fontsize=14, fontweight='bold')

for idx, col in enumerate(numeric_features):
    row = idx // 3
    col_idx = idx % 3
    
    sns.boxplot(data=df, x=target, y=col, ax=axes[row][col_idx], palette=['#e74c3c', '#2ecc71'])
    p_val = mw_results[idx]['p_value']
    sig_marker = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    axes[row][col_idx].set_title(f'{col}\np={p_val:.4f} {sig_marker}', fontsize=10)
    axes[row][col_idx].set_xlabel('0=Late, 1=On-Time')

plt.tight_layout()
plt.show()

## 5. Summary of Statistical Findings

In [ ]:
# Create a comprehensive summary table
print("\n" + "="*70)
print("  COMPREHENSIVE STATISTICAL SUMMARY")
print("="*70)

print("\n📊 CATEGORICAL FEATURES (Chi-Square Test):")
print("-"*50)
for result in chi_results:
    print(f"  {result['Feature']:<25} p={result['p-value']:.4f}  {result['Significant']}")

print("\n📈 NUMERIC FEATURES (Mann-Whitney U Test):")
print("-"*50)
for result in mw_results:
    sig = '✅ Yes' if result['p_value'] < 0.05 else '❌ No'
    print(f"  {result['Feature']:<25} p={result['p_value']:.4f}  {sig}")

# Count significant features
sig_cat = sum(1 for r in chi_results if r['p-value'] < 0.05)
sig_num = sum(1 for r in mw_results if r['p_value'] < 0.05)

print(f"\n📋 VERDICT:")
print(f"  {sig_cat} out of {len(chi_results)} categorical features are significant")
print(f"  {sig_num} out of {len(mw_results)} numeric features are significant")
print(f"\n  Features with p < 0.05 are statistically important for")
print(f"  predicting whether a delivery will be on time.")

## Key Takeaways

- **Skewness analysis** reveals which features have normal vs skewed distributions
- **Chi-Square test** identifies which categorical features influence delivery timeliness
- **Mann-Whitney U test** shows which numeric features differ between late and on-time deliveries
- Features with **p < 0.05** should be prioritized in the prediction model